In [0]:
!pip install yfinance

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StockPipelie").getOrCreate()
data = [(1, "Gustavo"), (2, "Copilot"), (3, "Databricks")]
df = spark.createDataFrame(data, ["id", "name"])
# df.show()


In [0]:
import yfinance as yf

# Example: Download data for Microsoft (MSFT)
df = yf.download(["PETR4.SA","VALE3.SA"],period="1y", interval="1d")
df.reset_index(inplace=True)

# Flatten column names (remove multi-index) 
# df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

# Convert to Spark Dataframe
spark_df = spark.createDataFrame(df)
spark_df.show(5)


In [0]:
import yfinance as yf

# Download multiple tickers
df = yf.download(["PETR4.SA","VALE3.SA"], period="1y", interval="1d")

# Stack the ticker level into rows
df_long = df.stack().reset_index().rename(columns={"level_2":"Ticker"})

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(df_long)

# Order by Ticker, Date
spark_df = spark_df.orderBy("Date", "Ticker")
spark_df.show(10)


In [0]:
from pyspark.sql.functions import col, lag
from pyspark.sql.window import Window

# Define window ordered by Date
windowSpec = Window.orderBy("Ticker", "Date")

# Add previous day's Close price
spark_df = spark_df.withColumn("PrevClose", lag("Close").over(windowSpec))

# Calculate daily return
spark_df = spark_df.withColumn("DailyReturn", (col("Close") - col("PrevClose")) / col("PrevClose"))

spark_df.select("Ticker", "Date", "Close", "PrevClose", "DailyReturn").show(10)